In [18]:
import pandas as pd
import ast

In [19]:
df = pd.read_csv('data\Family_Tasks_Dataset.csv')
print(len(df))
df.head()

100


,Member ID,Age,Skills,Availability,Task Preference,Task Name,Task Description,Task Duration,Task Difficulty,Priority Level,Task Skills Required,Assigned Status,Feedback
0,M001,50,"['Teaching', 'DIY', 'Cooking']",Afternoons,Physical,Fold laundry,Prepare a chocolate cake for dessert.,2 hours,Hard,Low,"['DIY', 'Organizing']",Unassigned,-
1,M002,24,"['Cleaning', 'DIY']",Afternoons,Indoor,Vacuum the living room,Repair the broken chair.,1 hour,Medium,High,"['Baking', 'Gardening']",Unassigned,Great job!
2,M003,11,"['Cleaning', 'Teaching']",Afternoons,Outdoor,Help with homework,Decorate the house for the party.,30 minutes,Easy,Low,"['Cleaning', 'Cooking']",Completed,Fantastic effort!
3,M004,51,"['Painting', 'Organizing', 'Teaching']",Afternoons,Physical,Fold laundry,Paint the backyard fence.,3 hours,Hard,High,['Teaching'],Completed,Could improve!
4,M005,47,"['Gardening', 'Cooking']",Evenings,Indoor,Paint the fence,Arrange books by size and genre.,1 hour,Medium,Medium,['Cooking'],Completed,Could improve!


In [20]:
df.isna().sum()

Member ID               0
Age                     0
Skills                  0
Availability            0
Task Preference         0
Task Name               0
Task Description        0
Task Duration           0
Task Difficulty         0
Priority Level          0
Task Skills Required    0
Assigned Status         0
Feedback                0
dtype: int64

In [21]:
df['Feedback'].fillna('No Feedback', inplace=True)


In [22]:
df['Skills'] = df['Skills'].apply(ast.literal_eval)
df['Task Skills Required'] = df['Task Skills Required'].apply(ast.literal_eval)

In [23]:
task_preference_mapping = {'Indoor': 0, 'Outdoor': 1, 'Physical': 2, 'Creative': 3}
df['Task Preference'] = df['Task Preference'].map(task_preference_mapping)

task_difficulty_mapping = {'Easy': 0, 'Medium': 1, 'Hard': 2}
df['Task Difficulty'] = df['Task Difficulty'].map(task_difficulty_mapping)

assigned_status_mapping = {'Unassigned': 0, 'Assigned': 1, 'Completed': 2}
df['Assigned Status'] = df['Assigned Status'].map(assigned_status_mapping)


In [24]:
df['Age Group'] = pd.cut(df['Age'], bins=[0, 12, 19, 59, 100], labels=['Child', 'Teen', 'Adult', 'Senior'])

In [25]:
df.head()

,Member ID,Age,Skills,Availability,Task Preference,Task Name,Task Description,Task Duration,Task Difficulty,Priority Level,Task Skills Required,Assigned Status,Feedback,Age Group
0,M001,50,"[Teaching, DIY, Cooking]",Afternoons,2,Fold laundry,Prepare a chocolate cake for dessert.,2 hours,2,Low,"[DIY, Organizing]",0,-,Adult
1,M002,24,"[Cleaning, DIY]",Afternoons,0,Vacuum the living room,Repair the broken chair.,1 hour,1,High,"[Baking, Gardening]",0,Great job!,Adult
2,M003,11,"[Cleaning, Teaching]",Afternoons,1,Help with homework,Decorate the house for the party.,30 minutes,0,Low,"[Cleaning, Cooking]",2,Fantastic effort!,Child
3,M004,51,"[Painting, Organizing, Teaching]",Afternoons,2,Fold laundry,Paint the backyard fence.,3 hours,2,High,[Teaching],2,Could improve!,Adult
4,M005,47,"[Gardening, Cooking]",Evenings,0,Paint the fence,Arrange books by size and genre.,1 hour,1,Medium,[Cooking],2,Could improve!,Adult


In [26]:
df['Task Duration'] = df['Task Duration'].str.extract('(\d+)').astype(float)
df.head()

,Member ID,Age,Skills,Availability,Task Preference,Task Name,Task Description,Task Duration,Task Difficulty,Priority Level,Task Skills Required,Assigned Status,Feedback,Age Group
0,M001,50,"[Teaching, DIY, Cooking]",Afternoons,2,Fold laundry,Prepare a chocolate cake for dessert.,2.0,2,Low,"[DIY, Organizing]",0,-,Adult
1,M002,24,"[Cleaning, DIY]",Afternoons,0,Vacuum the living room,Repair the broken chair.,1.0,1,High,"[Baking, Gardening]",0,Great job!,Adult
2,M003,11,"[Cleaning, Teaching]",Afternoons,1,Help with homework,Decorate the house for the party.,30.0,0,Low,"[Cleaning, Cooking]",2,Fantastic effort!,Child
3,M004,51,"[Painting, Organizing, Teaching]",Afternoons,2,Fold laundry,Paint the backyard fence.,3.0,2,High,[Teaching],2,Could improve!,Adult
4,M005,47,"[Gardening, Cooking]",Evenings,0,Paint the fence,Arrange books by size and genre.,1.0,1,Medium,[Cooking],2,Could improve!,Adult


In [27]:
from sklearn.preprocessing import MinMaxScaler
import joblib
scaler = MinMaxScaler()
df['Task Duration Scaled'] = scaler.fit_transform(df[['Task Duration']])
joblib.dump(scaler,'Duration_Scaled')

['Duration_Scaled']

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50)
task_description_tfidf = tfidf.fit_transform(df['Task Description'])
task_description_df = pd.DataFrame(task_description_tfidf.toarray(), columns=tfidf.get_feature_names_out())
df = pd.concat([df, task_description_df], axis=1)

In [29]:
joblib.dump(tfidf,'tfidf')

['tfidf']

In [30]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)